# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL and includes a detailed schema, record sets, and metadata. All references to dataset elements (record sets, fields) in this notebook use their `@id` for precision, in line with the Croissant specification.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We'll begin by loading the dataset metadata using `mlcroissant`. This gives us an overview of the data and the available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

Let's list available record sets, their fields, and their unique `@id` values from the Croissant metadata. These IDs are used for unambiguous reference throughout analysis.

In [ ]:
# Display record sets and their fields with @id

print("Record Sets available in the dataset:")
record_sets = []
for rs in metadata.record_sets:
    print(f"- Record Set: {rs.name} @id={rs.id}")
    record_sets.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} @id={field.id} (type: {field.data_type})")
    print("")

# For demonstration, preview the first record of each record set
for rs in metadata.record_sets:
    print(f"Example record from RecordSet {rs.id}:")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        if i == 0:
            print(rec)
            break
    print()

## 3. Data Extraction

We'll extract all available record sets defined by their `@id` into Pandas DataFrames. This enables flexible manipulation and analysis.

In [ ]:
# List all record set ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For demonstration, show all available record_set_ids and preview their columns
print("Available record sets and their columns:")
for rs_id, df in dataframes.items():
    print(f"\nRecord Set @id: {rs_id}")
    print(f"Columns: {list(df.columns)}")

# Let's pick the main tabular record set -- pick the one with the largest number of rows
main_rs_id = max(dataframes, key=lambda k: len(dataframes[k]))
print(f"\nMain Record Set ID selected for further analysis: {main_rs_id}")
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Explore the data, focusing on numeric and grouping variables. We'll apply typical preprocessing: filtering based on a numeric field, normalizing, and grouping/categorizing. All field names use their exact `@id` values as loaded from the schema.

In [ ]:
# Find likely numeric and grouping fields programmatically (by Croissant field metadata)

main_rs = None
for rs in metadata.record_sets:
    if rs.id == main_rs_id:
        main_rs = rs
        break

# Gather numeric and categorical fields
numeric_fields = [f.id for f in main_rs.fields if f.data_type and f.data_type.lower() in ["integer", "float", "number"]]
categorical_fields = [f.id for f in main_rs.fields if f.data_type and f.data_type.lower() in ["string", "text"]]

print("Numeric fields by @id:", numeric_fields)
print("Categorical/group fields by @id:", categorical_fields)

# If no numeric fields found, fallback: select columns that look numeric
df = dataframes[main_rs_id].copy()

if not numeric_fields:
    possible_numeric = df.select_dtypes(include=['number']).columns.tolist()
    if possible_numeric:
        numeric_field = possible_numeric[0]
    else:
        raise Exception("No numeric fields found for EDA.")
else:
    numeric_field = numeric_fields[0]

# Pick a grouping field
import numpy as np
group_field = None
for field in categorical_fields:
    n_uniques = df[field].nunique(dropna=True)
    if n_uniques > 1 and n_uniques < len(df)/2:
        group_field = field
        break
if group_field is None:
    group_field = df.columns[1] if len(df.columns) > 1 else df.columns[0]

print(f"\nWill filter and normalize field: {numeric_field} and group by: {group_field}")

# Remove records with missing or non-numeric value for filtering
df = df.copy()
if not np.issubdtype(df[numeric_field].dtype, np.number):
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].mean() if pd.notna(df[numeric_field].mean()) else 0
# Filter based on threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records where {numeric_field} > {threshold:.2f} (mean):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the chosen field
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
    print(f"\nAverage {numeric_field} grouped by {group_field} (showing first 5 groups):")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric variable and summarize its grouping using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Visualize grouped means, if grouping field is available
if group_field in filtered_df.columns and not filtered_df[group_field].isnull().all():
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, ci=None)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded rich clinical cancer survivor data using the Croissant standard and `mlcroissant`.
- All dataset elements were precisely referenced by their `@id`, ensuring reproducible and schema-anchored exploration.
- We identified key numeric and categorical variables, filtered and normalized records, and visualized core distributions.
- This workflow can be extended to further clinical or machine learning tasks, leveraging the FAIR and interoperable nature of Croissant datasets.